In [21]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [22]:
pd.set_option("display.width", 200)

IN_PATH = Path("../data/processed/tomat_harian.csv")
OUT_DIR = Path("../data/processed")
FIG_DIR = Path("../reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(IN_PATH, parse_dates=["tanggal"])
raw = raw.sort_values("tanggal").reset_index(drop=True)

In [23]:
raw

,tanggal,komoditas,satuan,harga,anomali
0,2023-01-01,Tomat,Kg,13428.571430,False
1,2023-01-02,Tomat,Kg,13857.142860,False
2,2023-01-03,Tomat,Kg,13428.571430,False
3,2023-01-04,Tomat,Kg,12285.714290,False
4,2023-01-05,Tomat,Kg,11571.428570,False
...,...,...,...,...,...
1330,2026-08-27,Tomat,Kg,9500.000000,False
1331,2026-08-28,Tomat,Kg,9833.333333,False
1332,2026-08-29,Tomat,Kg,9666.666667,False
1333,2026-08-30,Tomat,Kg,9666.666667,False


In [24]:
#  Pemeriksaan struktur 
print(f"Baris          : {len(raw):,}")
print(f"Rentang        : {raw['tanggal'].min().date()} s.d. {raw['tanggal'].max().date()}")
print(f"Duplikat tgl   : {raw['tanggal'].duplicated().sum()}")
print(f"Satuan         : {raw['satuan'].unique().tolist()}")
print(f"Harga <= 0     : {(raw['harga'] <= 0).sum()}")
print(f"Harga NaN      : {raw['harga'].isna().sum()}")
print(f"Anomali (flag) : {raw['anomali'].sum()}")

#  Frekuensi: bandingkan dengan kalender harian lengkap 
kalender = pd.date_range(raw["tanggal"].min(), raw["tanggal"].max(), freq="D")
tgl_hilang = kalender.difference(raw["tanggal"])

print(f"\nFrekuensi terinferensi : {pd.infer_freq(raw['tanggal'])}  (None = ada lubang)")
print(f"Hari seharusnya        : {len(kalender):,}")
print(f"Hari tidak ada         : {len(tgl_hilang)}")
print(f"Tanggal hilang         : {[d.strftime('%Y-%m-%d') for d in tgl_hilang]}")

Baris          : 1,335
Rentang        : 2023-01-01 s.d. 2026-08-31
Duplikat tgl   : 0
Satuan         : ['Kg']
Harga <= 0     : 0
Harga NaN      : 0
Anomali (flag) : 0

Frekuensi terinferensi : None  (None = ada lubang)
Hari seharusnya        : 1,339
Hari tidak ada         : 4
Tanggal hilang         : ['2025-03-31', '2026-04-28', '2026-04-29', '2026-04-30']


In [25]:
# Penangan lubang (missing dates) dengan interpolasi linear

# 1. Indeks harian lengkap: tanggal yang hilang menjadi baris NaN
s = raw.set_index("tanggal")[["komoditas", "satuan", "harga"]].asfreq("D")
s["komoditas"] = s["komoditas"].ffill()
s["satuan"] = s["satuan"].ffill()
s["diimputasi"] = s["harga"].isna()
s["harga_asli"] = s["harga"]

print("Hari diimputasi:", s.index[s["diimputasi"]].strftime("%Y-%m-%d").tolist())

# 2. Imputasi: interpolasi linear berbasis waktu (gap maks 3 hari)
s["harga_bersih"] = s["harga"].interpolate(method="time", limit=7, limit_area="inside")
assert s["harga_bersih"].isna().sum() == 0
assert s.index.freq == "D" and len(s) == 1339

# 3. Cek hasil imputasi di sekitar tiap gap
for t in s.index[s["diimputasi"]]:
    jendela = s.loc[t - pd.Timedelta(days=2): t + pd.Timedelta(days=2), ["harga_asli", "harga_bersih", "diimputasi"]]
    print(f"\n--- sekitar {t.date()} ---")
    print(jendela.round(0))

Hari diimputasi: ['2025-03-31', '2026-04-28', '2026-04-29', '2026-04-30']

--- sekitar 2025-03-31 ---
            harga_asli  harga_bersih  diimputasi
tanggal                                         
2025-03-29     11429.0       11429.0       False
2025-03-30     11429.0       11429.0       False
2025-03-31         NaN       11429.0        True
2025-04-01     11429.0       11429.0       False
2025-04-02     11429.0       11429.0       False

--- sekitar 2026-04-28 ---
            harga_asli  harga_bersih  diimputasi
tanggal                                         
2026-04-26     12286.0       12286.0       False
2026-04-27     12286.0       12286.0       False
2026-04-28         NaN       12214.0        True
2026-04-29         NaN       12143.0        True
2026-04-30         NaN       12071.0        True

--- sekitar 2026-04-29 ---
            harga_asli  harga_bersih  diimputasi
tanggal                                         
2026-04-27     12286.0       12286.0       False
2026-04-2

In [26]:
y = s["harga_bersih"].rename("y")
n = len(y)
i_train, i_val = int(n * 0.70), int(n * 0.85)
train, val, test = y.iloc[:i_train], y.iloc[i_train:i_val], y.iloc[i_val:]

for t in s.index[s["diimputasi"]]:
    bagian = "train" if t in train.index else "val" if t in val.index else "test"
    print(t.date(), "->", bagian)

print(f"\ntrain: {train.index[0].date()} s.d. {train.index[-1].date()} ({len(train)})")
print(f"val  : {val.index[0].date()} s.d. {val.index[-1].date()} ({len(val)})")
print(f"test : {test.index[0].date()} s.d. {test.index[-1].date()} ({len(test)})")

2025-03-31 -> train
2026-04-28 -> test
2026-04-29 -> test
2026-04-30 -> test

train: 2023-01-01 s.d. 2025-07-25 (937)
val  : 2025-07-26 s.d. 2026-02-11 (201)
test : 2026-02-12 s.d. 2026-08-31 (201)


In [27]:
# Dataset lengkap
s[["komoditas", "satuan", "harga_asli", "harga_bersih", "diimputasi"]] \
    .to_csv(OUT_DIR / "tomat_harian_bersih.csv", encoding="utf-8-sig")

# Split, dengan kolom flag supaya hari imputasi bisa dikeluarkan dari metrik
for nama, bag in [("train", train), ("val", val), ("test", test)]:
    out = bag.to_frame().join(s["diimputasi"])
    out.to_csv(OUT_DIR / f"tomat_{nama}.csv")

pd.DataFrame({"tanggal": s.index[s["diimputasi"]],
              "harga_imputasi": s.loc[s["diimputasi"], "harga_bersih"].round(0).values}) \
  .to_csv("../reports/tables/hari_diimputasi_tomat.csv", index=False)

In [28]:
test_df = pd.read_csv(OUT_DIR / "tomat_test.csv", index_col=0, parse_dates=True)
mask = ~test_df["diimputasi"]          # hanya hari dengan data asli
# mae = np.abs(test_df.loc[mask, "y"] - y_pred[mask]).mean()